# Spectral low-latency transport: reproducible evaluation

This notebook analyses compact, checked-in summaries produced by commit `69180e1`. The tests use WSL, private network namespaces and `tc netem`; they establish relative comparisons and delivery behaviour, **not** absolute AWS/ENA latency. The published CSVs retain every observation used here; full component logs are intentionally not versioned.

The supplied consumer measures end-to-end latency. On two hosts, do not interpret it until clock synchronization and its measured error are documented.

In [1]:
from pathlib import Path
import re
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure

output_notebook()
ROOT = Path('results')

def read_kv(path):
    return dict(line.strip().split('=', 1) for line in path.read_text().splitlines() if '=' in line)

def percentiles(path):
    result = {}
    for line in path.read_text().splitlines():
        match = re.match(r'\s*(p50|p99|p99\.9|p99\.99)\s*:\s*(\d+)', line)
        if match: result[match.group(1)] = int(match.group(2))
    return result


Loading BokehJS ...

## Speed-and-tail control

Six zero-loss runs compared frame-level XOR(4+1) with 1472-byte opportunistic packing against the one-frame-per-datagram control. Each measured 1,000,000 messages after a 500 ms warm-up at 100k messages/s; the four pipeline processes were pinned to separate CPUs.

Packing had p50 4.9–5.0 µs and p99.99 1.4–1.5 ms in all three repetitions. The unpacked control had lower median (4.5–4.7 µs) but far less stable tails, including a 33.4 ms p99.99 outlier. This supports packed `xor4` as the local **speed/tail candidate**, not as an AWS claim.

In [2]:
tail = pd.read_csv(ROOT / 'tail.csv').sort_values(['profile', 'repeat'])
tail


,profile,repeat,missing,p50_ns,p99_ns,p999_ns,p9999_ns
0,packed,1,0,5024,191721,697708,1400687
1,packed,2,0,4858,186604,610394,1459015
2,packed,3,0,4974,201097,731947,1517136
3,unpacked,1,0,4726,13675855,24993982,33375407
4,unpacked,2,0,4540,288373,775682,1358718
5,unpacked,3,0,4589,282841,870088,3106179


## Loss recovery

The matrix used 100,000 measured messages at 50k messages/s, 500 ms warm-up, three repetitions, a changed netem seed per repetition, and rotated profile order. `measurement_missing_total` is the residual gap after recovery and resequencing.

| Loss model | xor4 packed | xor4 unpacked | rs4_2 packed | packet_xor4 compact |
| --- | ---: | ---: | ---: | ---: |
| Independent 0.1% | 21.3 | 16.3 | 12.7 | 1.0 |
| Independent 0.5% | 108.3 | 96.0 | 91.3 | 15.0 |
| Gilbert-Elliott burst | 125.0 | 115.7 | 74.3 | 134.7 |

The evidence supports a conditional choice: packed `xor4` for the lowest-overhead speed profile; `packet_xor4` for independently scattered loss on the validated compact workload; and `rs4_2` for the tested burst model. No universal FEC winner is claimed.

In [3]:
recovery = pd.read_csv(ROOT / 'recovery.csv')
summary = recovery.groupby(['loss_model', 'profile'], as_index=False).agg(
    missing_mean=('missing', 'mean'), missing_min=('missing', 'min'),
    missing_max=('missing', 'max'), recovered_mean=('recovered', 'mean'))
summary

plot = figure(height=320, width=780, title='Residual message gaps after recovery',
              x_axis_label='profile', y_axis_label='mean missing messages',
              tools='pan,box_zoom,reset,save')
plot.vbar(x=list(range(len(summary))), top=list(summary.missing_mean), width=0.8,
          color='#4477aa')
plot.xaxis.ticker = list(range(len(summary)))
plot.xaxis.major_label_overrides = {i: f'{r.loss_model}\n{r.profile}' for i, r in summary.iterrows()}
show(plot)


## Fan-out and final decision

At 100k messages/s, all six zero-loss fan-out runs (two and three receivers; shared connected sender, `sendmmsg`, and one sender per receiver) delivered every one of 500,000 messages to every consumer. WSL p99.99 varied from 1.2 ms to 40.5 ms between topology variants, so it is not valid evidence to declare a fan-out syscall winner. Shared connected UDP remains the conservative topology.

Three independent links then confirmed the loss decision. At independent 0.1%/0.5%/0.1% loss, packet XOR left 0/27/4 residual gaps, XOR left 30/164/35, and RS left 16/130/21. Under the tested burst model, RS left 105/89/103 gaps, better than packet XOR's 162/146/179 and XOR's 147/139/180. All runs recorded zero sender socket drops.

The default submission candidate is full-frame packed `xor4` over connected UDP with `--reorder-wait-us auto`, because it has the least FEC overhead and the most stable local zero-loss tail among the directly compared XOR variants. For measured independent loss, switch both endpoints to `--fec packet_xor4 --wire-format compact --batch-bytes 1472`; for measured bursts, switch both to `--fec rs4_2 --wire-format full --batch-bytes 1472`. Re-run this matrix on AWS/Ubuntu/ENA before declaring absolute latency.

In [4]:
pd.read_csv(ROOT / 'fanout_loss.csv')


,run,receiver,loss_pct,missing,recovered,received
0,burst-packet_xor4,1,0.1,162,90,99838
1,burst-packet_xor4,2,0.1,146,117,99854
2,burst-packet_xor4,3,0.1,179,95,99821
3,burst-rs4_2,1,0.1,105,133,99895
4,burst-rs4_2,2,0.1,89,115,99911
5,burst-rs4_2,3,0.1,103,139,99897
6,burst-xor4,1,0.1,147,77,99853
7,burst-xor4,2,0.1,139,75,99861
8,burst-xor4,3,0.1,180,79,99820
9,random-packet_xor4,1,0.1,0,219,100000
